In [1]:
import yaml
import numpy as np
import zstandard as zstd
from pathlib import Path

from TraceSimulator import LongTraceSimulator
from trace_IO import *   # if you need anything from here

# -------------------
# Config
# -------------------
CONFIG_PATH = "/home/dwong/DELight_mtr/trigger_study/wk26/training_config.yaml"
DTYPE = np.float16
COMPRESSION_LEVEL = 15

SMOKE_ENERGY = 10.0
SMOKE_EVENTS = 8   # small number for quick test


def read_yaml_to_dict(file_path):
    with open(file_path, "r") as file:
        config_dict = yaml.safe_load(file)
    return config_dict


# -------------------
# Helpers (same pattern as your main code)
# -------------------
def _shuffle_bytes(arr: np.ndarray, dtype=DTYPE) -> bytes:
    a = np.asarray(arr, dtype=dtype)
    return a.view(np.uint8).reshape(-1, a.itemsize).T.tobytes()


def _unshuffle_bytes(data: bytes, dtype, shape) -> np.ndarray:
    itemsize = np.dtype(dtype).itemsize
    num_elements = int(np.prod(shape))
    reshaped = np.frombuffer(data, dtype=np.uint8).reshape(itemsize, num_elements).T
    unshuffled = reshaped.reshape(-1)
    return unshuffled.view(dtype).reshape(shape)


def save_traces_streaming(traces_iter, output_path: Path, compression_level=COMPRESSION_LEVEL):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cctx = zstd.ZstdCompressor(level=compression_level)
    with open(output_path, "wb") as f:
        with cctx.stream_writer(f) as zw:
            for tr in traces_iter:
                zw.write(_shuffle_bytes(tr))


def load_traces_from_zstd(path: Path, n_events: int, trace_shape, dtype=DTYPE) -> np.ndarray:
    """
    Minimal loader for smoke test: invert the shuffle and reshape.
    """
    itemsize = np.dtype(dtype).itemsize
    per_event_bytes = int(np.prod(trace_shape)) * itemsize
    expected_size = n_events * per_event_bytes

    dctx = zstd.ZstdDecompressor()
    with open(path, "rb") as f:
        compressed = f.read()
        decompressed = dctx.decompress(compressed, max_output_size=expected_size)

    if len(decompressed) != expected_size:
        raise ValueError(f"Decompressed size {len(decompressed)} != expected {expected_size}")

    traces = []
    for i in range(n_events):
        start = i * per_event_bytes
        end = start + per_event_bytes
        block = decompressed[start:end]
        tr = _unshuffle_bytes(block, dtype=dtype, shape=trace_shape)
        traces.append(tr)
    return np.stack(traces).astype(np.float32)


# -------------------
# Smoke test
# -------------------
def smoke_test():
    print("Loading config...")
    cfg = read_yaml_to_dict(CONFIG_PATH)
    lts = LongTraceSimulator(cfg)

    print("Running vectorized generate(E=[E]*N)...")
    E_vec = [SMOKE_ENERGY] * SMOKE_EVENTS

    # IMPORTANT: your real generate signature has more args; adapt if needed.
    ts, (x, y, z) = lts.generate(
        E=E_vec,
        x=None, y=None, z=None,
        type_recoil="NR",
        phonon_only=False,
        no_noise=False,
        quantize=True,
    )

    print("ts.shape:", ts.shape)
    print("positions shapes:", np.shape(x), np.shape(y), np.shape(z))

    # Expect: (SMOKE_EVENTS, n_LAMCAL, trace_samples)
    assert ts.shape[0] == SMOKE_EVENTS, "Bad number of events in ts"
    n_events, n_channels, n_samples = ts.shape
    print(f"n_events={n_events}, n_channels={n_channels}, n_samples={n_samples}")

    # Quick sanity expectations (you know these from your simulator)
    # For your long-trace setup, you expect n_channels=56, n_samples=150000
    # but we don't assert that hard in case config differs:
    # assert n_channels == 56
    # assert n_samples == 150000

    # ---- round-trip through zstd for these 8 events ----
    out_path = Path("smoke_traces_energy_10.zst")
    print(f"Writing {SMOKE_EVENTS} traces to {out_path}...")
    save_traces_streaming((ts[i] for i in range(SMOKE_EVENTS)), out_path)

    print("Loading back from zstd...")
    ts_loaded = load_traces_from_zstd(out_path, n_events=SMOKE_EVENTS,
                                      trace_shape=(n_channels, n_samples),
                                      dtype=DTYPE)
    print("ts_loaded.shape:", ts_loaded.shape)

    assert ts_loaded.shape == ts.shape, "Shape mismatch after zstd round-trip"

    # Compare a couple of events numerically (within quantization)
    for i in range(min(3, SMOKE_EVENTS)):
        diff = np.max(np.abs(ts_loaded[i].astype(np.float32) - ts[i].astype(np.float32)))
        print(f"max |diff| for event {i}:", diff)

    print("\nSmoke test completed successfully ✅")


if __name__ == "__main__":
    smoke_test()


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
Loading config...


/home/dwong/software/TraceSimulator/TraceSimulator/LongTraceSimulator.py:205: RuntimeWarning: overflow encountered in exp
  self.template = np.concatenate([(np.exp((xs - self.trigger_time) / self.tau_rise))[xs <= self.trigger_time], (np.exp(-(xs - self.trigger_time) / self.tau_decay))[xs > self.trigger_time]])


Running vectorized generate(E=[E]*N)...
ts.shape: (8, 56, 150000)
positions shapes: (8,) (8,) (8,)
n_events=8, n_channels=56, n_samples=150000
Writing 8 traces to smoke_traces_energy_10.zst...
Loading back from zstd...
ts_loaded.shape: (8, 56, 150000)
max |diff| for event 0: 0.0
max |diff| for event 1: 0.0
max |diff| for event 2: 0.0

Smoke test completed successfully ✅
